# Phase 5 — Final training of Model E and testing (in-dataset and cross-dataset)
* Architecture: Model E (1D-CNN + 2x ECA + GAP + skip branch + Dense), selected in the ablation study (Section 3.8.5)
* Data: features_v3 (Section 3.6.4); the complete final training split of each configuration
* Class imbalance: class weights w_c = N / (K * N_c), computed on the training split only — Kamal & Mashaly (2025)
* Adam, learning rate 1e-4, cross-entropy, batch size 128, up to 100 epochs, dropout 0.3 — Wahab et al. (2025)
* Early stopping: patience 5 on the loss of the complete validation split, best weights restored — Wahab et al. (2025)
* Test: Macro F1 (C. Wang et al., 2024), accuracy, weighted F1, per-class precision/recall/F1, confusion matrix,
  on the untouched test split
* Cross-dataset test (Cantone et al., 2024; Layeghy & Portmann): train on one dataset, test on the other.
  - only the 13 unified features of the merged configuration are common to both datasets
  - only the classes present in both datasets are used (Spoofing is absent from CICIDS2017) — Cantone et al. (2024)
  - the Min–Max scaling is refitted on the training dataset only, so the test dataset is never seen
  - each cross model is also tested on its own dataset's test split (in-domain baseline), so the drop
    from in-domain to cross-dataset performance can be reported (Layeghy & Portmann)
* Fault tolerance: after every epoch the model, the best weights and the early-stopping state are saved to Drive
  with atomic replacement; an interrupted run resumes from its last finished epoch. Re-run Cell 2 after any stop.
Outputs: outputs_v3/final_v1/ (results/, checkpoints/, models/, final_v1_log.txt, summary.json)

In [ ]:
# %% Cell 0 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %% Cell 1 — settings, model, helpers
import os, json, time, gc, math, platform
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

FEAT = '/content/drive/MyDrive/Thesis/outputs_v3/features_v3/'
OUT  = '/content/drive/MyDrive/Thesis/outputs_v3/final_v1/'
for sub in ('results/', 'checkpoints/', 'models/'):
    os.makedirs(OUT + sub, exist_ok=True)
MODEL = 'E'
SEED, LR, BATCH, EPOCHS, PATIENCE = 42, 1e-4, 128, 100, 5
N_CLASSES = 8
CLASS_NAMES = ['BENIGN', 'Botnet', 'BruteForce', 'DDoS', 'DoS', 'PortScan', 'Spoofing', 'WebAttack']
SRC = {'cicids': 'CICIDS2017', 'iot': 'CIC-IoT2023'}
RUNS = [  # (name, kind, train data, test data)
    ('final_config1_merged',     'in',    'config1_merged',     'config1_merged'),
    ('final_config2_cicids2017', 'in',    'config2_cicids2017', 'config2_cicids2017'),
    ('final_config3_ciciot2023', 'in',    'config3_ciciot2023', 'config3_ciciot2023'),
    ('cross_CICIDS2017_to_CIC-IoT2023', 'cross', SRC['cicids'], SRC['iot']),
    ('cross_CIC-IoT2023_to_CICIDS2017', 'cross', SRC['iot'], SRC['cicids']),
]
TOTAL_RUNS = len(RUNS)
LOGFILE = OUT + 'final_v1_log.txt'

def log(msg):
    line = time.strftime('%Y-%m-%d %H:%M:%S') + ' ' + msg
    print(line, flush=True)
    with open(LOGFILE, 'a') as f:                        # the full log is also kept on Drive
        f.write(line + '\n')

def hm(minutes):
    minutes = max(0.0, float(minutes))
    return f'{int(minutes // 60)} h {int(round(minutes % 60)):02d} min' if minutes >= 60 else f'{minutes:.1f} min'

def eca(x):
    c = x.shape[-1]
    w = layers.GlobalAveragePooling1D()(x)
    w = layers.Reshape((c, 1))(w)
    w = layers.Conv1D(1, 3, padding='same', use_bias=False, activation='sigmoid')(w)
    w = layers.Reshape((1, c))(w)
    return layers.Multiply()([x, w])

def build(variant, n_features, n_classes=N_CLASSES):
    """A: CNN+GAP | B: +ECA | C: +skip | D: +ECA +skip | E: +2xECA +skip (identical to the ablation study)"""
    inp = keras.Input((n_features, 1))
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    if variant == 'E':
        x = eca(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    if variant in ('B', 'D', 'E'):
        x = eca(x)
    z = layers.GlobalAveragePooling1D()(x)
    if variant in ('C', 'D', 'E'):
        s = layers.Dense(128, activation='relu')(layers.Flatten()(inp))
        z = layers.Concatenate()([z, s])
    z = layers.Dense(64, activation='relu')(z)
    z = layers.Dropout(0.3)(z)
    out = layers.Dense(n_classes, activation='softmax')(z)
    return keras.Model(inp, out, name=f'Model_{variant}')

def counts(y):
    c = np.bincount(y, minlength=N_CLASSES)
    return {CLASS_NAMES[i]: int(n) for i, n in enumerate(c) if n}

def class_weights(y):
    """w_c = N / (K * N_c) over the K classes present in the training data (Kamal & Mashaly, 2025)."""
    c = np.bincount(y, minlength=N_CLASSES); present = np.flatnonzero(c)
    return {int(k): float(len(y) / (len(present) * c[k])) for k in present}

def result_path(name):
    return OUT + f'results/{name}.json'

def ckpt_dir(name):
    return OUT + f'checkpoints/{name}/'

def n_done():
    return sum(os.path.exists(result_path(r[0])) for r in RUNS)

def save_atomic_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def val_metrics(model, Xv, yv):
    """Loss, accuracy, Macro F1 and per-class F1 on the complete validation split (one prediction pass)."""
    p = model.predict(Xv, batch_size=8192, verbose=0)
    loss = float(-np.mean(np.log(np.clip(p[np.arange(len(yv)), yv], 1e-7, 1.0))))   # unweighted cross-entropy
    yp = p.argmax(1); labels = np.unique(yv)
    f1c = f1_score(yv, yp, average=None, labels=labels, zero_division=0)
    return loss, float(accuracy_score(yv, yp)), float(f1c.mean()), {CLASS_NAMES[k]: round(float(v), 3) for k, v in zip(labels, f1c)}

def evaluate(model, X, y, labels=None):
    """Metrics on a test split; labels = the classes over which Macro F1 is averaged (default: those present in y)."""
    t0 = time.time()
    pred = model.predict(X, batch_size=8192, verbose=0).argmax(1)
    sec = time.time() - t0
    labels = np.unique(y) if labels is None else np.asarray(labels)
    pr, rc, f1c, sup = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    return dict(macro_f1=float(f1c.mean()), accuracy=float(accuracy_score(y, pred)),
                weighted_f1=float(f1_score(y, pred, average='weighted', labels=labels, zero_division=0)),
                per_class={CLASS_NAMES[k]: dict(precision=float(p_), recall=float(r_), f1=float(f_), support=int(s_))
                           for k, p_, r_, f_, s_ in zip(labels, pr, rc, f1c, sup)},
                confusion_matrix=confusion_matrix(y, pred, labels=labels).tolist(),
                labels=[CLASS_NAMES[k] for k in labels], test_rows=int(len(y)),
                predicted_outside_labels=int((~np.isin(pred, labels)).sum()),
                inference_seconds=round(sec, 2), inference_ms_per_1000=round(1000 * 1000 * sec / max(1, len(y)), 3))

def log_eval(prefix, r):
    log(f"{prefix}: Macro F1 {r['macro_f1']:.4f} | accuracy {r['accuracy']:.4f} | weighted F1 {r['weighted_f1']:.4f} | "
        f"{r['test_rows']:,} test rows | inference {r['inference_ms_per_1000']:.3f} ms per 1,000 records")
    log('   per class (precision / recall / F1 / support): ' + '; '.join(
        f"{k} {v['precision']:.3f}/{v['recall']:.3f}/{v['f1']:.3f}/{v['support']:,}" for k, v in r['per_class'].items()))
    if r['predicted_outside_labels']:
        log(f"   {r['predicted_outside_labels']:,} records were predicted as a class outside the evaluated classes (counted as errors)")

# ---------- data
def load_config(cfg, split):
    X = np.load(FEAT + cfg + f'/X_{split}.npy').astype(np.float32, copy=False)
    y = np.load(FEAT + cfg + f'/y_{split}.npy').astype(np.int64)
    return X, y

def merged_raw(split, source):
    """Rows of one dataset in the merged configuration, with the merged Min–Max scaling (and log transform) undone."""
    sp = json.load(open(FEAT + 'config1_merged/spec.json'))
    X, y = load_config('config1_merged', split)
    s = np.load(FEAT + f'config1_merged/source_{split}.npy')
    keep = s == source
    X, y = X[keep].astype(np.float64), y[keep]
    mn, mx = np.array(sp['train_min']), np.array(sp['train_max'])
    use_log = np.array([f in sp['log_features'] for f in sp['features']])
    lo = np.where(use_log, np.log1p(np.maximum(mn, 0)), mn)
    hi = np.where(use_log, np.log1p(np.maximum(mx, 0)), mx)
    rng = np.where(hi > lo, hi - lo, 1.0)
    X = X * rng + lo
    X[:, use_log] = np.expm1(X[:, use_log])
    return X, y, sp

def fit_scaling(train, feats, wide_range):
    """Same rule as in Section 3.6.3 (log(1+x) for wide-range non-negative features, then Min–Max), fitted on train only."""
    mn, mx = train.min(0), train.max(0)
    use_log = ((mx - mn) > wide_range) & (mn >= 0)
    lo = np.where(use_log, np.log1p(np.maximum(mn, 0)), mn)
    hi = np.where(use_log, np.log1p(np.maximum(mx, 0)), mx)
    rng = np.where(hi > lo, hi - lo, 1.0)
    def tf_(x):
        x = np.array(x, copy=True)
        x[:, use_log] = np.log1p(np.maximum(x[:, use_log], 0))
        return ((x - lo) / rng).astype(np.float32)
    return tf_, [f for f, u in zip(feats, use_log) if u]

def get_data(name, kind, a, b):
    if kind == 'in':
        Xtr, ytr = load_config(a, 'train'); Xv, yv = load_config(a, 'val'); Xte, yte = load_config(a, 'test')
        extra = {}
        if a == 'config1_merged':                         # test rows of each dataset, for the per-dataset breakdown
            extra['source_test'] = np.load(FEAT + a + '/source_test.npy')
        return Xtr, ytr, Xv, yv, {'own': (Xte, yte)}, None, extra
    # cross-dataset: train/val of dataset a, test of dataset b (and of a, as in-domain baseline)
    Rtr, ytr, sp = merged_raw('train', a); Rv, yv, _ = merged_raw('val', a)
    Rte_a, yte_a, _ = merged_raw('test', a); Rte_b, yte_b, _ = merged_raw('test', b)
    shared = sorted(set(np.unique(ytr)) & set(np.unique(yte_b)))
    dropped = sorted((set(np.unique(ytr)) | set(np.unique(yte_b))) - set(shared))
    log(f'{name}: classes in both datasets {[CLASS_NAMES[k] for k in shared]} | left out (present in only one): {[CLASS_NAMES[k] for k in dropped]}')
    def keep(R, y):
        m = np.isin(y, shared); return R[m], y[m]
    Rtr, ytr = keep(Rtr, ytr); Rv, yv = keep(Rv, yv); Rte_a, yte_a = keep(Rte_a, yte_a); Rte_b, yte_b = keep(Rte_b, yte_b)
    tf_, log_feats = fit_scaling(Rtr, sp['features'], sp['wide_range_threshold'])
    Xtr, Xv, Xa, Xb = tf_(Rtr), tf_(Rv), tf_(Rte_a), tf_(Rte_b)
    out_rng = float(((Xb < 0) | (Xb > 1)).any(1).mean())
    log(f'{name}: scaling refitted on the {a} training rows only | log(1+x) features: {log_feats}')
    log(f'{name}: {100 * out_rng:.1f}% of the {b} test rows have at least one feature outside the training range [0, 1] '
        f'(kept as they are, no clipping)')
    del Rtr, Rv, Rte_a, Rte_b; gc.collect()
    return Xtr, ytr, Xv, yv, {'in_domain': (Xa, yte_a), 'cross': (Xb, yte_b)}, shared, dict(scaling_log_features=log_feats,
                                                                                         target_rows_outside_train_range=out_rng)

class ProgressCheckpoint(keras.callbacks.Callback):
    """Detailed progress (every 10% of an epoch, with time left), complete-validation evaluation, early stopping
    (patience 5, best weights kept) and an atomic checkpoint after every epoch."""
    def __init__(self, tag, d, Xv, yv, steps, state):
        super().__init__()
        self.tag, self.d, self.Xv, self.yv, self.steps, self.state = tag, d, Xv, yv, steps, state
        self.marks = {max(1, int(steps * q / 10)) for q in range(1, 10)}
        self.t_run = time.time() - state['minutes'] * 60
    def on_epoch_begin(self, epoch, logs=None):
        self.t_ep, self.cur = time.time(), epoch + 1
        log(f"{self.tag} | epoch {self.cur} started ({self.steps:,} steps of {BATCH})")
    def on_train_batch_end(self, batch, logs=None):
        if batch + 1 in self.marks:
            el = (time.time() - self.t_ep) / 60; frac = (batch + 1) / self.steps
            log(f"{self.tag} | epoch {self.cur} | step {batch + 1:,}/{self.steps:,} ({100 * frac:.0f}%) | "
                f"running loss {logs['loss']:.4f} | {hm(el)} elapsed, about {hm(el / frac - el)} left in this epoch")
    def on_epoch_end(self, epoch, logs=None):
        st, d = self.state, self.d
        t_train = (time.time() - self.t_ep) / 60
        vloss, vacc, vf1, vpc = val_metrics(self.model, self.Xv, self.yv)
        improved = vloss < st['best']
        if improved:
            self.model.save_weights(d + 'best.tmp.weights.h5'); os.replace(d + 'best.tmp.weights.h5', d + 'best.weights.h5')
            st.update(best=vloss, best_ep=epoch + 1, wait=0)
        else:
            st['wait'] += 1
        st.update(epoch=epoch + 1, minutes=(time.time() - self.t_run) / 60)
        st.setdefault('history', []).append(dict(epoch=epoch + 1, loss=round(float(logs['loss']), 5), val_loss=round(vloss, 5),
                                                 val_acc=round(vacc, 5), val_macro_f1=round(vf1, 5)))
        self.model.save(d + 'last.tmp.keras'); os.replace(d + 'last.tmp.keras', d + 'last.keras')
        save_atomic_json(st, d + 'state.json')
        ep_min = st['minutes'] / st['epoch']
        log(f"{self.tag} | epoch {epoch + 1} done | train {t_train:.1f} min + validation {(time.time() - self.t_ep) / 60 - t_train:.1f} min | "
            f"loss {logs['loss']:.4f} | val_loss {vloss:.4f}{' (best)' if improved else ''} | val_acc {vacc:.4f} | val_macroF1 {vf1:.4f}")
        log(f"{self.tag} |   val F1 per class: {vpc}")
        log(f"{self.tag} |   best epoch {st['best_ep']} | no improvement {st['wait']}/{PATIENCE} | run time {hm(st['minutes'])} | "
            f"at most {PATIENCE - st['wait']} more epochs without improvement (about {hm((PATIENCE - st['wait']) * ep_min)}) | checkpoint saved")
        if st['wait'] >= PATIENCE:
            log(f"{self.tag} | early stopping: no improvement for {PATIENCE} epochs; best epoch {st['best_ep']} will be restored")
            self.model.stop_training = True

log('=' * 100)
log(f'Final training v1 (Model {MODEL}) | TensorFlow {tf.__version__} | Python {platform.python_version()} | GPU: '
    f'{[g.name for g in tf.config.list_physical_devices("GPU")] or "none"}')
log(f'settings: complete training splits, lr {LR}, batch {BATCH}, max epochs {EPOCHS}, patience {PATIENCE}, '
    f'class weights N/(K*N_c), complete validation split | {n_done()} of {TOTAL_RUNS} runs already finished')

In [ ]:
# %% Cell 2 — final training and testing (resumable: re-run this cell after any interruption)
T0 = time.time()
for i, (name, kind, a, b) in enumerate(RUNS, 1):
    if os.path.exists(result_path(name)):
        log(f'run {i} of {TOTAL_RUNS} ({name}): already finished, skipping'); continue
    t_load = time.time()
    Xtr, ytr, Xv, yv, tests, shared, extra = get_data(name, kind, a, b)
    assert np.isfinite(Xtr).all() and np.isfinite(Xv).all(), 'non-finite values in the data'
    Xtr, Xv = Xtr[..., None], Xv[..., None]
    cw = class_weights(ytr); steps = int(math.ceil(len(ytr) / BATCH))
    head = f'--- run {i} of {TOTAL_RUNS}: {name} (Model {MODEL}) ---'
    log(head)
    log(f'{name}: training {len(ytr):,} rows, {Xtr.shape[1]} features | validation {len(yv):,} rows | ' +
        ' | '.join(f'test "{k}" {len(v[1]):,} rows' for k, v in tests.items()) + f' | loaded in {time.time() - t_load:.0f}s')
    log(f'{name}: training class counts {counts(ytr)}')
    log(f'{name}: class weights ' + str({CLASS_NAMES[k]: round(v, 4) for k, v in cw.items()}))
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
    d = ckpt_dir(name); os.makedirs(d, exist_ok=True)
    if os.path.exists(d + 'state.json') and os.path.exists(d + 'last.keras'):
        state = json.load(open(d + 'state.json'))
        model = keras.models.load_model(d + 'last.keras')
        ep_model = int(keras.ops.convert_to_numpy(model.optimizer.iterations)) // steps
        log(f"{name} --- RESUMING after epoch {state['epoch']} (best epoch {state['best_ep']}, "
            f"no improvement {state['wait']}/{PATIENCE}, {hm(state['minutes'])} already trained) ---")
        if ep_model == state['epoch'] + 1:            # interrupted between saving the model and saving the state
            log(f'{name} | the saved model is one epoch ahead of the saved state; evaluating it to bring the state up to date')
            vloss = val_metrics(model, Xv, yv)[0]
            if vloss < state['best']:
                model.save_weights(d + 'best.tmp.weights.h5'); os.replace(d + 'best.tmp.weights.h5', d + 'best.weights.h5')
                state.update(best=vloss, best_ep=ep_model, wait=0)
            else:
                state['wait'] += 1
            state['epoch'] = ep_model; save_atomic_json(state, d + 'state.json')
        elif ep_model != state['epoch']:
            log(f"{name} | note: optimizer step count corresponds to epoch {ep_model}, state says {state['epoch']}; continuing from the state")
    else:
        state = dict(epoch=0, best=float('inf'), best_ep=0, wait=0, minutes=0.0, history=[])
        model = build(MODEL, Xtr.shape[1])
        model.compile(optimizer=keras.optimizers.Adam(LR), loss='sparse_categorical_crossentropy')
    params = int(sum(np.prod(v.shape) for v in model.trainable_weights))
    log(f'{name} | {params:,} trainable parameters | {steps:,} steps per epoch')
    if not (state['wait'] >= PATIENCE or state['epoch'] >= EPOCHS):
        model.fit(Xtr, ytr, batch_size=BATCH, epochs=EPOCHS, initial_epoch=state['epoch'], verbose=0, shuffle=True,
                  class_weight=cw, callbacks=[ProgressCheckpoint(name, d, Xv, yv, steps, state)])
    model.load_weights(d + 'best.weights.h5')                      # best epoch (early stopping)
    model.save(OUT + f'models/{name}.tmp.keras'); os.replace(OUT + f'models/{name}.tmp.keras', OUT + f'models/{name}.keras')
    size_kb = os.path.getsize(OUT + f'models/{name}.keras') / 1024
    res = dict(run=name, kind=kind, train_data=a, test_data=b, model=MODEL, epochs=state['epoch'], best_epoch=state['best_ep'],
               train_minutes=round(state['minutes'], 2), params=params, model_file_kb=round(size_kb, 1),
               train_rows=int(len(ytr)), class_weights={CLASS_NAMES[k]: v for k, v in cw.items()},
               history=state.get('history', []), tests={})
    for k, (Xt, yt) in tests.items():
        r = evaluate(model, Xt[..., None], yt, labels=shared)
        res['tests'][k] = r
        log_eval(f'{name} RESULT [{k} test: {b if k == "cross" else a}]', r)
    if 'source_test' in extra:                          # merged model: per-dataset breakdown of the merged test split
        Xt, yt = tests['own']; s = extra['source_test']
        for src in (SRC['cicids'], SRC['iot']):
            m = s == src
            r = evaluate(model, Xt[m][..., None], yt[m])
            res['tests'][f'merged_test_{src}'] = r
            log_eval(f'{name} RESULT [merged test split, {src} rows only]', r)
    if kind == 'cross':
        ind, crs = res['tests']['in_domain']['macro_f1'], res['tests']['cross']['macro_f1']
        res['macro_f1_drop_percent'] = round(100 * (ind - crs) / ind, 2) if ind > 0 else None
        res.update({k: v for k, v in extra.items()})
        log(f"{name}: in-domain Macro F1 {ind:.4f} -> cross-dataset Macro F1 {crs:.4f} | drop {res['macro_f1_drop_percent']}%")
    save_atomic_json(res, result_path(name))
    log(f"{name}: {res['epochs']} epochs (best {res['best_epoch']}) | trained in {hm(res['train_minutes'])} | "
        f"model file {size_kb:.0f} KB | saved to models/{name}.keras")
    log(f'   finished {n_done()} of {TOTAL_RUNS} runs | this session running for {hm((time.time() - T0) / 60)}')
    del model, Xtr, ytr, Xv, yv, tests; gc.collect()
log(f'final training v1: {n_done()} of {TOTAL_RUNS} runs finished')

In [ ]:
# %% Cell 3 — summary tables
rows = {r[0]: json.load(open(result_path(r[0]))) for r in RUNS if os.path.exists(result_path(r[0]))}
print(f'{len(rows)} of {TOTAL_RUNS} runs finished\n')
t1 = [dict(configuration=n.replace('final_', ''), macro_f1=r['tests']['own']['macro_f1'], accuracy=r['tests']['own']['accuracy'],
           weighted_f1=r['tests']['own']['weighted_f1'], epochs=r['epochs'], best_epoch=r['best_epoch'],
           params=r['params'], model_kb=r['model_file_kb'], train_minutes=r['train_minutes'])
      for n, r in rows.items() if r['kind'] == 'in']
if t1:
    print('In-dataset test (Model E, complete test split):'); print(pd.DataFrame(t1).round(4).to_string(index=False))
m = rows.get('final_config1_merged')
if m:
    print('\nMerged model, test split of each dataset:')
    print(pd.DataFrame([dict(dataset=k.replace('merged_test_', ''), macro_f1=v['macro_f1'], accuracy=v['accuracy'], rows=v['test_rows'])
                        for k, v in m['tests'].items() if k.startswith('merged_test_')]).round(4).to_string(index=False))
t2 = [dict(train=r['train_data'], test=r['test_data'], classes=', '.join(r['tests']['cross']['labels']),
           in_domain_macro_f1=r['tests']['in_domain']['macro_f1'], cross_macro_f1=r['tests']['cross']['macro_f1'],
           drop_percent=r['macro_f1_drop_percent'], cross_accuracy=r['tests']['cross']['accuracy'])
      for n, r in rows.items() if r['kind'] == 'cross']
if t2:
    print('\nCross-dataset test (13 unified features, classes present in both datasets):')
    print(pd.DataFrame(t2).round(4).to_string(index=False))
    for n, r in rows.items():
        if r['kind'] == 'cross':
            print(f"\n{n} — per class on the other dataset (precision / recall / F1):")
            for k, v in r['tests']['cross']['per_class'].items():
                print(f"   {k:11s} {v['precision']:.3f} / {v['recall']:.3f} / {v['f1']:.3f}   ({v['support']:,} rows)")
save_atomic_json(dict(in_dataset=t1, cross_dataset=t2), OUT + 'summary.json')